# GTEx tissue validation — triangulating 3 independent resources

Mirrors `06_global_alignment.ipynb` / `07_tissue_cellmarker_validation.ipynb`: same SHAP selections (top1, cumulative25), same FDR-significance "tissue_correct" criterion, same decreasing-raw-Z gene selection, same `clusterProfiler::enricher` ORA — run once per resource, then combined into one comparison so a tissue's validation isn't resting on a single resource's idiosyncrasies.

Three independent ground-truth resources (none used as a training prior for this CLAMP model, none GTEx-bulk-derived):

1. **CellMarker 2024** — rerun here for direct comparison (same result as `07_tissue_cellmarker_validation.ipynb`).
2. **MSigDB C8** — cell-type signatures, already used elsewhere in this pipeline for Panel C ORA.
3. **PanglaoDB** — organ-annotated marker table (official release, since the live site blocks `curl`/bot access and Enrichr's mirror lacks tissue labels).

In [1]:
library(here)
library(dplyr)
library(tidyr)
library(stringr)
library(clusterProfiler)


SHAP_DIR <- here('output', '03_model_biology', '01_gtex',
                 '02_rf_kmeans', '00_LV_importance_kmeans',
                 'gtex_feature_importance_kmeans_binary_shap')
OUT_DIR  <- here('output', '03_model_biology', '01_gtex', '02_rf_kmeans', '08_tissue_validation_multi_resource')
dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)

CLAMP_RDS         <- here('output', '01_model_building', '02_gtex', '01_CLAMP', 'CLAMPfull.rds')
CELLMARKER_FILE   <- here('data', 'pathways', 'CellMarker_2024.txt')
C8_GMT_FILE       <- here('data', 'pathways', 'c8.all.v2026.1.Hs.symbols.gmt')
PANGLAODB_FILE    <- here('data', 'pathways', 'PanglaoDB_markers_27_Mar_2020.tsv.gz')

N_LVS_PER_TISSUE <- 1L
CUMULATIVE_PCT   <- 25
TOP_GENE_PCT     <- 0.01
FDR_THRESH       <- 0.05
MIN_GS           <- 10
MAX_GS           <- 500

normalize_text <- function(x) {
    x <- tolower(x)
    x <- gsub('[^a-z0-9]+', ' ', x)
    stringr::str_squish(x)
}

here() starts at /home/msubirana/Documents/pivlab/clamp-analyses




Attaching package: ‘dplyr’




The following objects are masked from ‘package:stats’:

    filter, lag




The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




clusterProfiler v4.14.0 Learn more at https://yulab-smu.top/contribution-knowledge-mining/

Please cite:

S Xu, E Hu, Y Cai, Z Xie, X Luo, L Zhan, W Tang, Q Wang, B Liu, R Wang,
W Xie, T Wu, L Xie, G Yu. Using clusterProfiler to characterize
multiomics data. Nature Protocols. 2024, 19(11):3292-3320




Attaching package: ‘clusterProfiler’




The following object is masked from ‘package:stats’:

    filter




## SHAP-based LV selection (top1, cumulative25) — same as `06_global_alignment.ipynb`

In [2]:
shap_all <- read.delim(file.path(SHAP_DIR, 'all_shap_positive.tsv'),
                       stringsAsFactors = FALSE, check.names = FALSE)

selected_lvs_top1 <- shap_all %>%
    dplyr::arrange(Tissue, Rank) %>%
    dplyr::group_by(Tissue) %>%
    dplyr::slice_head(n = N_LVS_PER_TISSUE) %>%
    dplyr::ungroup() %>%
    dplyr::rename(LV = Feature) %>%
    dplyr::select(Tissue, LV, Rank, Mean_SHAP_Tissue, Cumulative_SHAP, Cumulative_Percent)

selected_lvs_cum <- shap_all %>%
    dplyr::arrange(Tissue, Rank) %>%
    dplyr::group_by(Tissue) %>%
    dplyr::mutate(
        reaches_thresh = Cumulative_Percent >= CUMULATIVE_PCT,
        cutoff_rank = if (any(reaches_thresh)) min(Rank[reaches_thresh]) else max(Rank)
    ) %>%
    dplyr::filter(Rank <= cutoff_rank) %>%
    dplyr::ungroup() %>%
    dplyr::rename(LV = Feature) %>%
    dplyr::select(Tissue, LV, Rank, Mean_SHAP_Tissue, Cumulative_SHAP, Cumulative_Percent)

selections <- list(top1 = selected_lvs_top1, cumulative25 = selected_lvs_cum)

cat('Tissues:', dplyr::n_distinct(selected_lvs_top1$Tissue), '\n')
for (nm in names(selections)) {
    cat(sprintf('  [%s] selected LV/tissue rows: %d, unique LVs: %d\n',
                nm, nrow(selections[[nm]]), dplyr::n_distinct(selections[[nm]]$LV)))
}

Tissues: 23 


  [top1] selected LV/tissue rows: 23, unique LVs: 21
  [cumulative25] selected LV/tissue rows: 79, unique LVs: 69


In [3]:
clamp <- readRDS(CLAMP_RDS)
Z_full <- as.matrix(clamp$Z)
rm(clamp)

all_selected_lvs <- unique(unlist(lapply(selections, function(df) df$LV)))
selected_unique_lvs <- intersect(all_selected_lvs, colnames(Z_full))
Z <- Z_full[, selected_unique_lvs, drop = FALSE]
rm(Z_full)
universe_genes <- rownames(Z)
n_top_genes <- max(1L, ceiling(TOP_GENE_PCT * nrow(Z)))

top_genes_per_lv <- lapply(colnames(Z), function(lv) {
    vals <- Z[, lv]
    universe_genes[order(vals, decreasing = TRUE)[seq_len(n_top_genes)]]
})
names(top_genes_per_lv) <- colnames(Z)

cat('Z:', nrow(Z), 'genes x', ncol(Z), 'unique LVs\n')
cat('Top genes per LV:', n_top_genes, '\n')

Z: 21613 genes x 69 unique LVs


Top genes per LV: 217 


## Reusable functions

Run once per resource: build `term2gene_df` + `tissue_matched_names` (many-to-many — a term can count as ground truth for more than one of our GTEx tissues when they share a keyword, e.g. both Heart sub-regions), run ORA per LV, then summarise tissue-level correctness exactly like `06_global_alignment.ipynb`'s `run_alignment_summary`.

In [4]:
run_resource_ora <- function(term2gene_df, top_genes_per_lv, universe_genes) {
    run_one <- function(genes, universe) {
        res <- tryCatch(
            clusterProfiler::enricher(
                gene          = genes,
                universe      = universe,
                TERM2GENE     = term2gene_df,
                pAdjustMethod = 'BH',
                pvalueCutoff  = 1,
                qvalueCutoff  = 1,
                minGSSize     = MIN_GS,
                maxGSSize     = MAX_GS
            ),
            error = function(e) NULL
        )
        if (is.null(res)) return(NULL)
        df <- as.data.frame(res)
        if (nrow(df) == 0) return(NULL)
        df
    }

    ora_list <- lapply(names(top_genes_per_lv), function(lv) {
        df <- run_one(top_genes_per_lv[[lv]], universe_genes)
        if (is.null(df)) return(NULL)
        df$LV <- lv
        df
    })

    ora_all <- do.call(rbind, Filter(Negate(is.null), ora_list))
    if (is.null(ora_all)) return(data.frame())
    rownames(ora_all) <- NULL
    ora_all %>% dplyr::select(LV, ID, Description, GeneRatio, BgRatio, pvalue, p.adjust, qvalue, geneID, Count)
}

run_alignment_summary <- function(selected_lvs, ora_all, tissue_matched_names, out_dir) {
    dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)
    sig_ora <- ora_all %>% dplyr::filter(LV %in% selected_lvs$LV, p.adjust < FDR_THRESH)

    detail <- selected_lvs %>%
        dplyr::rowwise() %>%
        dplyr::mutate(
            own_terms = list(tissue_matched_names[[Tissue]]),
            n_sig_terms = sum(sig_ora$LV == LV),
            n_true_terms = sum(sig_ora$LV == LV & sig_ora$ID %in% unlist(own_terms)),
            tissue_correct = n_true_terms > 0,
            best_true_padj = {
                x <- sig_ora$p.adjust[sig_ora$LV == LV & sig_ora$ID %in% unlist(own_terms)]
                if (length(x) == 0) NA_real_ else min(x, na.rm = TRUE)
            },
            matched_terms = paste(sig_ora$ID[sig_ora$LV == LV & sig_ora$ID %in% unlist(own_terms)], collapse = ' | ')
        ) %>%
        dplyr::ungroup() %>%
        dplyr::select(-own_terms)

    tissue_summary <- detail %>%
        dplyr::group_by(Tissue) %>%
        dplyr::summarise(
            n_selected_lvs = dplyr::n(),
            tissue_correct = any(tissue_correct),
            correct_lvs = paste(LV[tissue_correct], collapse = ';'),
            best_true_padj = if (all(is.na(best_true_padj))) NA_real_ else min(best_true_padj, na.rm = TRUE),
            .groups = 'drop'
        ) %>%
        dplyr::mutate(
            has_coverage = vapply(Tissue, function(t) length(tissue_matched_names[[t]]) > 0, logical(1)),
            correct_score = as.integer(tissue_correct)
        )

    final_summary <- data.frame(
        n_tissues = nrow(tissue_summary),
        n_tissues_with_coverage = sum(tissue_summary$has_coverage),
        n_tissues_correct = sum(tissue_summary$tissue_correct),
        pct_tissue_correct = 100 * mean(tissue_summary$tissue_correct),
        pct_tissue_correct_of_covered = 100 * sum(tissue_summary$tissue_correct) / sum(tissue_summary$has_coverage),
        stringsAsFactors = FALSE
    )

    write.csv(detail, file.path(out_dir, 'alignment_detail.csv'), row.names = FALSE)
    write.csv(tissue_summary, file.path(out_dir, 'alignment_summary.csv'), row.names = FALSE)
    write.csv(final_summary, file.path(out_dir, 'alignment_final_pct.csv'), row.names = FALSE)

    list(detail = detail, tissue_summary = tissue_summary, final_summary = final_summary)
}
matched_names_to_df <- function(tissue_matched_names) {
    rows <- do.call(rbind, lapply(names(tissue_matched_names), function(t) {
        terms <- tissue_matched_names[[t]]
        if (length(terms) == 0) return(NULL)
        data.frame(tissue = t, term = terms, stringsAsFactors = FALSE)
    }))
    if (is.null(rows)) data.frame(tissue = character(0), term = character(0)) else rows
}


## Resource 1: CellMarker 2024

Same keyword table as `07_tissue_cellmarker_validation.ipynb` (19 tissues in scope; Pituitary/Vagina have zero CellMarker coverage, the two non-anatomical "tissues" — cultured fibroblasts, EBV-transformed lymphocytes — have no cell-line tag to match).

CellMarker 2024 set names follow a clean `<Cell Type> <Tissue> Human` convention (unlike C8's free-text study names), so all 1134 Human-suffixed names were read end to end to check the keyword table for gaps -- the same exhaustive approach used for C8 below, rather than only guessing keywords and trusting the count. Five confirmed misses were added (`EXTRA_CELLMARKER_MATCHES`, next cell): `Colorectum`-tagged terms for Colon - Transverse, `Airway`/`Bronch*`-tagged terms for Lung, an explicit (not regex) list of bare `... Muscle Human` terms for Muscle - Skeletal (excluding `Cardiomyocyte Heart Muscle`, which is cardiac), `Cardiac Atrium` for Heart - Atrial Appendage only (not Left Ventricle), and `Intestinal` for Small Intestine - Terminal Ileum. Pituitary and Vagina remain genuinely uncovered. Nerve - Tibial, Cells - Cultured fibroblasts, and Cells - EBV-transformed lymphocytes are NOT dead ends, though: Schwann cells (peripheral nerve's resident glial cell), 'Fibroblast Skin' (GTEx's cultured-fibroblast sample source), and blood-derived B cell/plasma cell terms (EBV immortalizes mature peripheral-blood B cells into the lymphoblastoid cell lines this "tissue" actually is) are all legitimate matches, added explicitly below.

Every resource's term→tissue assignment (`*_tissue_matched_names`) is also exported as an explicit, reviewable CSV (`*_term_tissue_assignment.csv`) rather than left implicit in code -- for PanglaoDB this is exhaustive by construction (built directly from the official `organ` column, no keyword guessing involved). The downstream criterion is unchanged: an LV counts as `tissue_correct` if at least one term assigned to its true tissue is FDR<0.05 significant among all tested terms (`run_alignment_summary`'s `n_true_terms > 0`).

In [5]:
read_gmt_human_suffix <- function(filename) {
    lines <- readLines(filename, warn = FALSE)
    gmt <- list()
    for (line in lines) {
        sp <- strsplit(line, '\t')[[1]]
        if (length(sp) < 3) next
        name <- sp[1]
        if (!endsWith(name, ' Human')) next
        genes <- sp[3:length(sp)]
        genes <- genes[nzchar(genes)]
        if (length(genes) > 0) gmt[[name]] <- genes
    }
    gmt
}

cellmarker_t2g <- read_gmt_human_suffix(CELLMARKER_FILE)

cellmarker_tissue_keywords <- c(
    'Adrenal Gland'                       = 'Adrenal',
    'Artery - Tibial'                     = '\\bArtery\\b',
    'Colon - Transverse'                  = '\\bColon\\b',
    'Esophagus - Mucosa'                  = 'Esophag',
    'Heart - Atrial Appendage'            = '\\bHeart\\b',
    'Heart - Left Ventricle'              = '\\bHeart\\b',
    'Kidney - Cortex'                     = '\\bKidney\\b',
    'Liver'                               = '\\bLiver\\b',
    'Lung'                                = '\\bLung\\b',
    'Muscle - Skeletal'                   = 'Skeletal Muscle',
    'Nerve - Tibial'                      = '\\bNerve\\b',
    'Ovary'                               = 'Ovar',
    'Pancreas'                            = 'Pancrea',
    'Prostate'                            = 'Prostate',
    'Small Intestine - Terminal Ileum'    = 'Intestine|Ileum',
    'Stomach'                             = '\\bStomach\\b',
    'Testis'                              = '\\bTesti(s|cular)',
    'Thyroid'                             = 'Thyroid',
    'Whole Blood'                         = '\\bBlood\\b'
)

# `search_names` defaults to `term_names` but can be a transformed copy (e.g. underscores
# replaced with spaces) so \b word-boundary patterns work against underscore-delimited
# names like C8's 'AIZARANI_LIVER_C10_MVECS_1' (underscore counts as a word character, so
# \bLIVER\b never matches '_LIVER_' literally).
build_tissue_matches <- function(term_names, tissue_keywords, search_names = term_names) {
    out <- lapply(tissue_keywords, function(pattern) term_names[grepl(pattern, search_names, ignore.case = TRUE)])
    names(out) <- names(tissue_keywords)
    out
}

cellmarker_term_names <- names(cellmarker_t2g)
cellmarker_tissue_matched_names <- build_tissue_matches(cellmarker_term_names, cellmarker_tissue_keywords)

# Manual additions found by reading all 1134 Human CellMarker term names end to end (not
# just keyword-guessing) -- these needed special handling that a single regex per tissue
# can't express cleanly:
#  - 'Colorectum' terms don't contain the substring 'Colon', so they were missed entirely
#    even though colorectum biology includes colon.
#  - 'Airway'/'Bronchial Vessel'/'Bronchiole'/'Bronchus' terms are part of the lung/
#    respiratory tract but don't contain 'Lung'.
#  - Bare '... Muscle Human' terms (Endothelial Cell, Fibroblast, Mesenchymal Stem Cell,
#    Myoblast, Stem Cell) were missed since only 'Skeletal Muscle' was in the pattern --
#    added explicitly by exact name (not regex) so 'Cardiomyocyte Heart Muscle' (cardiac,
#    not skeletal) is never swept in by accident.
#  - 'Cardiac Atrium' is atrium-specific, so it's added only to Heart - Atrial Appendage,
#    not Heart - Left Ventricle (previously both shared one identical 'Heart' keyword).
#  - 'Intestinal' (adjective form) wasn't caught by the 'Intestine' substring pattern.
EXTRA_CELLMARKER_MATCHES <- list(
    'Colon - Transverse' = cellmarker_term_names[grepl('Colorectum', cellmarker_term_names, ignore.case = TRUE)],
    'Lung' = cellmarker_term_names[grepl('Airway|Bronch', cellmarker_term_names, ignore.case = TRUE)],
    'Small Intestine - Terminal Ileum' = cellmarker_term_names[grepl('Intestinal', cellmarker_term_names, ignore.case = TRUE)],
    'Muscle - Skeletal' = cellmarker_term_names[cellmarker_term_names %in% c(
        'Endothelial Cell Muscle Human', 'Fibroblast Muscle Human',
        'Mesenchymal Stem Cell Muscle Human', 'Myoblast Muscle Human',
        'Stem Cell Muscle Human'
    )],
    'Heart - Atrial Appendage' = cellmarker_term_names[grepl('Cardiac Atrium', cellmarker_term_names, ignore.case = TRUE)],
    # Nerve - Tibial: Schwann cells are the actual resident glial cell of peripheral nerve
    # (myelinating/ensheathing axons) -- the single closest CellMarker analog to a nerve biopsy.
    'Nerve - Tibial' = cellmarker_term_names[grepl('Schwann Cell Skin', cellmarker_term_names, ignore.case = TRUE)],
    # Cells - Cultured fibroblasts: GTEx's "cultured fibroblasts" sample is specifically
    # skin punch biopsy fibroblasts grown in vitro -- 'Fibroblast Skin Human' is the direct match.
    'Cells - Cultured fibroblasts' = cellmarker_term_names[grepl('^Fibroblast Skin Human$', cellmarker_term_names)],
    # Cells - EBV-transformed lymphocytes: these are lymphoblastoid cell lines (LCLs) --
    # EBV preferentially immortalizes mature peripheral-blood B cells, so blood-derived
    # B cell/plasma cell/plasmablast terms are the correct ground truth (not B cells from
    # other organs, since LCLs are specifically blood-derived).
    'Cells - EBV-transformed lymphocytes' = cellmarker_term_names[
        grepl('B Cell|Plasma Cell|Plasmablast', cellmarker_term_names, ignore.case = TRUE) &
        grepl('Blood', cellmarker_term_names, ignore.case = TRUE)
    ]
)
for (t in names(EXTRA_CELLMARKER_MATCHES)) {
    cellmarker_tissue_matched_names[[t]] <- union(cellmarker_tissue_matched_names[[t]], EXTRA_CELLMARKER_MATCHES[[t]])
}

cellmarker_term2gene_df <- do.call(rbind, lapply(cellmarker_term_names, function(term) {
    data.frame(term = term, gene = cellmarker_t2g[[term]], stringsAsFactors = FALSE)
}))

cellmarker_assignment_df <- matched_names_to_df(cellmarker_tissue_matched_names)
write.csv(cellmarker_assignment_df, file.path(OUT_DIR, 'cellmarker_term_tissue_assignment.csv'), row.names = FALSE)

cellmarker_coverage <- data.frame(
    Tissue   = names(cellmarker_tissue_matched_names),
    N_terms  = vapply(cellmarker_tissue_matched_names, length, integer(1))
) %>% dplyr::arrange(N_terms)
cellmarker_coverage

,Tissue,N_terms
,<chr>,<int>
Cells - Cultured fibroblasts,Cells - Cultured fibroblasts,1
Nerve - Tibial,Nerve - Tibial,2
Thyroid,Thyroid,3
Artery - Tibial,Artery - Tibial,4
Adrenal Gland,Adrenal Gland,5
Muscle - Skeletal,Muscle - Skeletal,8
Prostate,Prostate,8
Esophagus - Mucosa,Esophagus - Mucosa,12
Cells - EBV-transformed lymphocytes,Cells - EBV-transformed lymphocytes,12


In [6]:
cellmarker_ora_all <- run_resource_ora(cellmarker_term2gene_df, top_genes_per_lv, universe_genes)
write.csv(cellmarker_ora_all, file.path(OUT_DIR, 'cellmarker_ora_per_lv.csv'), row.names = FALSE)

results_cellmarker <- lapply(names(selections), function(nm) {
    run_alignment_summary(selections[[nm]], cellmarker_ora_all, cellmarker_tissue_matched_names,
                           file.path(OUT_DIR, 'cellmarker', nm))
})
names(results_cellmarker) <- names(selections)

dplyr::bind_rows(lapply(names(results_cellmarker), function(nm) {
    results_cellmarker[[nm]]$final_summary %>% dplyr::mutate(analysis = nm, .before = 1)
}))

analysis,n_tissues,n_tissues_with_coverage,n_tissues_correct,pct_tissue_correct,pct_tissue_correct_of_covered
<chr>,<int>,<int>,<int>,<dbl>,<dbl>
top1,23,21,12,52.17391,57.14286
cumulative25,23,21,15,65.21739,71.42857


## CellMarker: all significant pathways per LV (top1 vs. cumulative25)

Two tables, one row per LV: every CellMarker term significant at FDR<0.05 for that LV, not just the terms matching its own tissue (unlike `alignment_detail.csv` above, which only reports `matched_terms` restricted to the LV's own tissue). This shows the full enrichment picture per LV -- useful for seeing what else (if anything) an LV's top genes are enriched for besides its assigned tissue.

In [7]:
build_lv_pathway_table <- function(selected_lvs, ora_all) {
    lv_tissue_map <- selected_lvs %>%
        dplyr::group_by(LV) %>%
        dplyr::summarise(Tissue = paste(unique(Tissue), collapse = '; '), .groups = 'drop')

    sig_by_lv <- ora_all %>%
        dplyr::filter(LV %in% lv_tissue_map$LV, p.adjust < FDR_THRESH) %>%
        dplyr::arrange(LV, p.adjust) %>%
        dplyr::group_by(LV) %>%
        dplyr::summarise(
            n_sig_pathways = dplyr::n(),
            sig_pathways = paste(sprintf('%s (FDR=%.2g)', ID, p.adjust), collapse = ' | '),
            .groups = 'drop'
        )

    lv_tissue_map %>%
        dplyr::left_join(sig_by_lv, by = 'LV') %>%
        dplyr::mutate(
            n_sig_pathways = tidyr::replace_na(n_sig_pathways, 0L),
            sig_pathways = tidyr::replace_na(sig_pathways, '')
        ) %>%
        dplyr::arrange(Tissue)
}

cellmarker_lv_pathways_top1 <- build_lv_pathway_table(selections$top1, cellmarker_ora_all)
cellmarker_lv_pathways_cum25 <- build_lv_pathway_table(selections$cumulative25, cellmarker_ora_all)

write.csv(cellmarker_lv_pathways_top1, file.path(OUT_DIR, 'cellmarker_lv_significant_pathways_top1.csv'), row.names = FALSE)
write.csv(cellmarker_lv_pathways_cum25, file.path(OUT_DIR, 'cellmarker_lv_significant_pathways_cumulative25.csv'), row.names = FALSE)

cat('--- top1: one row per LV, all significant CellMarker terms ---\n')
print(cellmarker_lv_pathways_top1, n = Inf)
cat('\n--- cumulative25: one row per LV, all significant CellMarker terms ---\n')
print(cellmarker_lv_pathways_cum25, n = Inf)

--- top1: one row per LV, all significant CellMarker terms ---


# A tibble: 21 × 4
   LV    Tissue                                     n_sig_pathways sig_pathways 
   <chr> <chr>                                               <int> <chr>        
 1 LV183 Adrenal Gland                                           2 "Leydig Cell…
 2 LV563 Artery - Tibial                                         0 ""           
 3 LV546 Cells - Cultured fibroblasts; Whole Blood              11 "Mesenchymal…
 4 LV21  Cells - EBV-transformed lymphocytes; Liver              8 "Liver Bud H…
 5 LV562 Colon - Transverse                                      9 "Intestinal …
 6 LV84  Esophagus - Mucosa                                      7 "Basal Cell …
 7 LV420 Heart - Atrial Appendage                                1 "Cardiomyocy…
 8 LV505 Heart - Left Ventricle                                  2 "Cardiomyocy…
 9 LV229 Kidney - Cortex                                         7 "Nephron Epi…
10 LV26  Lung                                                    4 "Epithelial …
11 LV137 


--- cumulative25: one row per LV, all significant CellMarker terms ---


# A tibble: 69 × 4
   LV    Tissue                                      n_sig_pathways sig_pathways
   <chr> <chr>                                                <int> <chr>       
 1 LV183 Adrenal Gland                                            2 "Leydig Cel…
 2 LV3   Adrenal Gland; Ovary                                     4 "Naive CD4+…
 3 LV37  Artery - Tibial                                          0 ""          
 4 LV387 Artery - Tibial                                          4 "Pit Progen…
 5 LV563 Artery - Tibial                                          0 ""          
 6 LV90  Artery - Tibial; Cells - EBV-transformed l…              0 ""          
 7 LV111 Cells - Cultured fibroblasts                            24 "MKI67+ Pro…
 8 LV363 Cells - Cultured fibroblasts                             0 ""          
 9 LV91  Cells - Cultured fibroblasts                           128 "Microglial…
10 LV546 Cells - Cultured fibroblasts; Whole Blood               11 "Mesenchyma…
11 LV116 

## Resource 2: MSigDB C8

C8 set names embed tissue context as free text (e.g. `AIZARANI_LIVER_C11_HEPATOCYTES_1`, `DESCARTES_FETAL_ADRENAL_CHROMAFFIN_CELLS`). Keyword matching alone isn't enough here: inspecting the full matched list per tissue (dumped to `/tmp/c8_tissue_matches_full.txt` during development) showed that many "matches" are **generic hematopoietic/immune passenger cell types that recur near-identically across almost every fetal organ** in the DESCARTES atlas — erythroblasts, myeloid/lymphoid cells, megakaryocytes, generic stromal/mesothelial cells, Schwann cells, and circulating immune cells (B/T/NK cells, macrophages, monocytes, dendritic cells, mast cells, neutrophils). These reflect blood cells passing through or developing within a vascularized fetal organ, not that organ's own resident biology — keeping them would let, say, any blood-related LV "match" Adrenal Gland, Heart, Kidney, Liver, Lung, etc. simultaneously, which defeats the purpose of an independent, tissue-*specific* check.

These generic passenger types are filtered out explicitly (`is_generic_passenger()` below), with two named exceptions kept as genuinely organ-iconic resident immune populations: Kupffer cells (liver-resident macrophages) and alveolar macrophages (lung-resident). Organ-specific vascular/endothelial cell types are kept even when the cell-type label itself looks generic (e.g. "vascular endothelial cells"), because they were profiled from that specific organ's tissue and organ-specific endothelial heterogeneity is real, well-documented biology (e.g. liver sinusoidal endothelial cells vs. lung capillary aerocytes are transcriptionally very different).

Generic `MUSCLE` over-matches unrelated smooth-muscle contexts (heart, eye, intestine, placenta), so skeletal muscle uses a more specific pattern. `GAO_LARGE_INTESTINE_*` (large intestine = colon) and `GAO_SMALL_INTESTINE_*` are separate study prefixes that a single generic `INTESTINE` keyword would wrongly lump together under one tissue -- split explicitly below, with the unqualified `DESCARTES_FETAL_INTESTINE_*` sets shared across both since they don't specify large vs. small. `Artery - Tibial`'s only "match" (`TRAVAGLINI_LUNG_ARTERY_CELL`) is a lung-atlas artery cell, not systemic arterial tissue, and all `BLOOD` matches are fetal/cord-blood progenitor sets (plus `HAY_BONE_MARROW_*`, which is bone marrow, not blood) -- both stay excluded entirely. Nerve, Prostate, Testis, and Thyroid have zero C8 coverage even after a broader re-check (SCIATIC/GANGLI/AXON/MYELIN for nerve; PROST/HENRY for prostate; SPERMAT/SERTOLI/LEYDIG/GERMLINE for testis -- note naive "TESTI" substring search is contaminated by matching inside "in-TESTI-ne"; THYRO/FOLLICULAR_EPITH for thyroid, where the only FOLLICULAR hit was a bone-marrow B-cell subtype, unrelated).

A later targeted re-check (direct greps against the raw GMT for artery/aorta/coronary/vascular/endothelial, testis/sperm/gonad/germ/sertoli/leydig, thyroid/follicular, blood/pbmc/leukocyte, prostate) confirmed Testis, Thyroid, Whole Blood, and Prostate genuinely have zero C8 matches, but found one real miss for `Artery - Tibial`: 3 terms explicitly labeled arterial endothelial cells (`TRAVAGLINI_LUNG_ARTERY_CELL`, `HE_LIM_SUN_FETAL_LUNG_C3_ARTERIAL_ENDOTHELIAL_CELL`, `HE_LIM_SUN_FETAL_LUNG_C3_GRIA2_POS_ARTERIAL_ENDO_CELL`). These are added explicitly (`EXTRA_C8_MATCHES`, next cell) since arterial endothelial specification (arterial vs. venous/capillary/lymphatic identity) is a conserved transcriptional program across vascular beds, unlike the organ-specific endothelial sets that stay excluded.

Three more genuine matches, found by the same reasoning (a cell type's identity can be conserved across organ-of-origin even when C8's passenger filter would normally exclude it as generic): Schwann cells for Nerve - Tibial (peripheral nerve's resident glial cell -- here NOT filtered as a passenger, since for this tissue it IS the target identity), non-activated fibroblast terms for Cells - Cultured fibroblasts (GTEx's sample is literally cultured skin fibroblasts), and mature (non-precursor) B cell terms for Cells - EBV-transformed lymphocytes (these are lymphoblastoid cell lines -- EBV-immortalized mature B cells).

In [8]:
read_gmt_standard <- function(filename) {
    lines <- readLines(filename, warn = FALSE)
    gmt <- list()
    for (line in lines) {
        sp <- strsplit(line, '\t')[[1]]
        if (length(sp) < 3) next
        genes <- sp[3:length(sp)]
        genes <- genes[nzchar(genes)]
        if (length(genes) > 0) gmt[[sp[1]]] <- genes
    }
    gmt
}

c8_t2g <- read_gmt_standard(C8_GMT_FILE)

c8_tissue_keywords <- c(
    'Adrenal Gland'                       = 'ADRENAL',
    'Esophagus - Mucosa'                  = 'ESOPHAG',
    'Heart - Atrial Appendage'            = 'HEART',
    'Heart - Left Ventricle'              = 'HEART',
    'Kidney - Cortex'                     = 'KIDNEY',
    'Liver'                               = 'LIVER',
    'Lung'                                = 'LUNG',
    'Muscle - Skeletal'                   = 'SKELETAL MUSCLE|FETAL MUSCLE ',
    'Ovary'                               = 'OVAR',
    'Pancreas'                            = 'PANCREA',
    # GAO_LARGE_INTESTINE_* (colon) vs GAO_SMALL_INTESTINE_* were previously both being
    # caught by a single generic 'INTESTINE' keyword and credited only to Small Intestine --
    # large intestine IS the colon, so it needs its own tissue. The unqualified
    # DESCARTES_FETAL_INTESTINE_*/DESCARTES_MAIN_FETAL_INTESTINAL_* sets don't specify
    # large vs small, so they're shared ground truth for both (same principle as Heart's
    # two sub-regions sharing the generic 'HEART' keyword).
    'Colon - Transverse'                  = 'LARGE INTESTINE|DESCARTES.*INTESTIN',
    # BUSSLINGER_DUODENAL_* (duodenum = proximal small intestine) was missing entirely --
    # found by reading every one of the 866 C8 set names rather than guessing keywords.
    'Small Intestine - Terminal Ileum'    = 'SMALL INTESTINE|DESCARTES.*INTESTIN|BUSSLINGER DUODENAL',
    # BUSSLINGER_GASTRIC_* (gastric = stomach) was also missing entirely, same way.
    'Stomach'                             = 'STOMACH|BUSSLINGER GASTRIC'
)

# Generic hematopoietic/immune/connective-tissue "passenger" cell types that recur
# near-identically across many organs (see markdown above) -- excluded everywhere except
# the two named organ-iconic resident immune populations.
GENERIC_PASSENGER_PATTERN <- paste(
    'ERYTHROBLAST', 'ERYTHROCYTE', 'RETICULOCYTE', 'HEMATOPOIETIC_STEM_CELL',
    'EMBRYONIC_RED_BLOOD', 'MYELOID_CELL', 'LYMPHOID_CELL', 'IMMUNE_CELL',
    'MEGAKARYOCYTE', 'PLATELET', 'MESOTHELIAL_CELL', 'SCHWANN_CELL',
    '(^|_)B_CELL', '(^|_)T_CELL', 'NK_CELL', 'NATURAL_KILLER', 'MACROPHAGE',
    'MONOCYTE', 'DENDRITIC_CELL', 'MAST_CELL', 'BASOPHIL', 'NEUTROPHIL',
    'GRANULOCYTE', 'GMP_CELL', '(^|_)HSC(_|$)', 'STROMAL_CELL', 'INTERSTITIUM',
    sep = '|'
)
GENERIC_PASSENGER_ALLOWLIST <- c('KUPFFER', 'ALVEOLAR_MACROPHAGE')

# Properly vectorized: per-element allowlist/generic checks, not a single any()-collapsed
# scalar (an earlier version used any() across the whole vector, which meant a single
# 'KUPFFER' anywhere in a tissue's matched list silently kept ALL of that tissue's names).
is_generic_passenger <- function(names_vec) {
    is_allowed <- Reduce(`|`, lapply(GENERIC_PASSENGER_ALLOWLIST, function(a) grepl(a, names_vec, ignore.case = TRUE)))
    is_generic <- grepl(GENERIC_PASSENGER_PATTERN, names_vec, ignore.case = TRUE)
    is_generic & !is_allowed
}

c8_term_names <- names(c8_t2g)
c8_term_names_spaced <- gsub('_', ' ', c8_term_names)  # underscore -> space (see build_tissue_matches note)
c8_tissue_matched_raw <- build_tissue_matches(c8_term_names, c8_tissue_keywords, c8_term_names_spaced)

# Apply the generic-passenger filter and print what got dropped, per tissue, for inspection.
c8_tissue_matched_names <- lapply(c8_tissue_matched_raw, function(names_vec) {
    names_vec[!is_generic_passenger(names_vec)]
})

for (tissue in names(c8_tissue_matched_raw)) {
    dropped <- setdiff(c8_tissue_matched_raw[[tissue]], c8_tissue_matched_names[[tissue]])
    cat(sprintf('%s: %d kept, %d dropped as generic passenger\n',
                tissue, length(c8_tissue_matched_names[[tissue]]), length(dropped)))
    if (length(dropped) > 0) cat('   dropped:', paste(dropped, collapse = ', '), '\n')
}

# Targeted grep audit of the raw C8 file for artery/aorta/coronary/vascular/endothelial,
# testis/sperm/gonad/germ/sertoli/leydig, thyroid/follicular, blood/pbmc/leukocyte, and
# prostate terms confirmed Testis, Thyroid, Whole Blood, and Prostate genuinely have zero
# C8 matches (no germ/gonad/Sertoli/Leydig term exists at all; the only 'FOLLICULAR' hit is
# a bone-marrow B-cell subtype unrelated to thyroid; all 'blood' hits are fetal/cord-blood
# progenitor populations, developmentally distinct from adult whole blood). Artery - Tibial
# had one genuine miss: 3 terms explicitly labeled arterial endothelial cells. These were
# previously excluded as lung-atlas-derived, but arterial endothelial specification
# (arterial vs. venous/capillary/lymphatic identity, e.g. the Notch/Hey1/Dll4 program,
# GJA4/GJA5, BMX) is a conserved transcriptional program across vascular beds, not organ-
# specific the way hepatocyte/cardiomyocyte identity is -- so these count as legitimate
# ground truth for systemic artery tissue, unlike the unlabeled organ-specific
# vascular/endothelial sets (e.g. DESCARTES_FETAL_LIVER_VASCULAR_ENDOTHELIAL_CELLS) which
# stay excluded since they're not specifically arterial.
EXTRA_C8_MATCHES <- list(
    'Artery - Tibial' = c(
        'TRAVAGLINI_LUNG_ARTERY_CELL',
        'HE_LIM_SUN_FETAL_LUNG_C3_ARTERIAL_ENDOTHELIAL_CELL',
        'HE_LIM_SUN_FETAL_LUNG_C3_GRIA2_POS_ARTERIAL_ENDO_CELL'
    ),
    # Nerve - Tibial: Schwann cells are peripheral nerve's defining resident glial cell --
    # previously only ever treated as a generic passenger when found inside OTHER organs
    # (heart/muscle/adrenal/lung), but here they're the actual target identity, so every
    # SCHWANN-labeled C8 term counts (no passenger filter needed for this one).
    'Nerve - Tibial' = c8_term_names[grepl('SCHWANN', c8_term_names, ignore.case = TRUE)],
    # Cells - Cultured fibroblasts: 'vanilla' (quiescent, non-activated) fibroblast terms --
    # excluding myofibroblast/lipofibroblast (specialized contractile/lipid-storing subtypes)
    # and cancer-associated/'-like' fibroblasts (disease-activated states), since GTEx's
    # cultured fibroblasts are ordinary skin-derived fibroblasts, not any of those variants.
    'Cells - Cultured fibroblasts' = c8_term_names[
        grepl('FIBROBLAST', c8_term_names, ignore.case = TRUE) &
        !grepl('MYOFIBROBLAST|LIPOFIBROBLAST|CANCER_ASSOCIATED|FIBROBLAST_LIKE', c8_term_names, ignore.case = TRUE)
    ],
    # Cells - EBV-transformed lymphocytes: EBV preferentially immortalizes mature B cells
    # into lymphoblastoid cell lines (LCLs) -- excluding pre-B/pro-B developmental precursor
    # stages, which are transcriptionally distinct immature stages, not what gets transformed.
    'Cells - EBV-transformed lymphocytes' = c8_term_names[
        grepl('B_CELL|B_LYMPHOCYTE', c8_term_names, ignore.case = TRUE) &
        !grepl('PRE_B|PRO_B', c8_term_names, ignore.case = TRUE)
    ]
)
for (t in names(EXTRA_C8_MATCHES)) {
    c8_tissue_matched_names[[t]] <- union(c8_tissue_matched_names[[t]], EXTRA_C8_MATCHES[[t]])
}

c8_term2gene_df <- do.call(rbind, lapply(c8_term_names, function(term) {
    data.frame(term = term, gene = c8_t2g[[term]], stringsAsFactors = FALSE)
}))

c8_assignment_df <- matched_names_to_df(c8_tissue_matched_names)
write.csv(c8_assignment_df, file.path(OUT_DIR, 'c8_term_tissue_assignment.csv'), row.names = FALSE)

c8_coverage <- data.frame(
    Tissue  = names(c8_tissue_matched_names),
    N_terms = vapply(c8_tissue_matched_names, length, integer(1))
) %>% dplyr::arrange(N_terms)
c8_coverage

Adrenal Gland: 6 kept, 6 dropped as generic passenger
   dropped: DESCARTES_FETAL_ADRENAL_ERYTHROBLASTS, DESCARTES_FETAL_ADRENAL_LYMPHOID_CELLS, DESCARTES_FETAL_ADRENAL_MEGAKARYOCYTES, DESCARTES_FETAL_ADRENAL_MYELOID_CELLS, DESCARTES_FETAL_ADRENAL_SCHWANN_CELLS, DESCARTES_FETAL_ADRENAL_STROMAL_CELLS 
Esophagus - Mucosa: 8 kept, 1 dropped as generic passenger
   dropped: BUSSLINGER_ESOPHAGEAL_DENDRITIC_CELLS 
Heart - Atrial Appendage: 32 kept, 9 dropped as generic passenger
   dropped: CUI_DEVELOPING_HEART_C7_MAST_CELL, CUI_DEVELOPING_HEART_C8_MACROPHAGE, CUI_DEVELOPING_HEART_C9_B_T_CELL, DESCARTES_FETAL_HEART_ERYTHROBLASTS, DESCARTES_FETAL_HEART_LYMPHOID_CELLS, DESCARTES_FETAL_HEART_MEGAKARYOCYTES, DESCARTES_FETAL_HEART_MYELOID_CELLS, DESCARTES_FETAL_HEART_SCHWANN_CELLS, DESCARTES_FETAL_HEART_STROMAL_CELLS 
Heart - Left Ventricle: 32 kept, 9 dropped as generic passenger
   dropped: CUI_DEVELOPING_HEART_C7_MAST_CELL, CUI_DEVELOPING_HEART_C8_MACROPHAGE, CUI_DEVELOPING_HEART_C9_B_T_CELL, 

,Tissue,N_terms
,<chr>,<int>
Artery - Tibial,Artery - Tibial,3
Adrenal Gland,Adrenal Gland,6
Esophagus - Mucosa,Esophagus - Mucosa,8
Nerve - Tibial,Nerve - Tibial,10
Muscle - Skeletal,Muscle - Skeletal,14
Cells - EBV-transformed lymphocytes,Cells - EBV-transformed lymphocytes,15
Cells - Cultured fibroblasts,Cells - Cultured fibroblasts,18
Ovary,Ovary,20
Pancreas,Pancreas,21


In [9]:
c8_ora_all <- run_resource_ora(c8_term2gene_df, top_genes_per_lv, universe_genes)
write.csv(c8_ora_all, file.path(OUT_DIR, 'c8_ora_per_lv.csv'), row.names = FALSE)

results_c8 <- lapply(names(selections), function(nm) {
    run_alignment_summary(selections[[nm]], c8_ora_all, c8_tissue_matched_names,
                           file.path(OUT_DIR, 'c8', nm))
})
names(results_c8) <- names(selections)

dplyr::bind_rows(lapply(names(results_c8), function(nm) {
    results_c8[[nm]]$final_summary %>% dplyr::mutate(analysis = nm, .before = 1)
}))

analysis,n_tissues,n_tissues_with_coverage,n_tissues_correct,pct_tissue_correct,pct_tissue_correct_of_covered
<chr>,<int>,<int>,<int>,<dbl>,<dbl>
top1,23,17,14,60.86957,82.35294
cumulative25,23,17,15,65.21739,88.23529


## Resource 3: PanglaoDB

Official marker table (`organ` is a clean controlled-vocabulary column, no regex needed). "GI tract" and "Reproductive" are excluded — too coarse to assign to a single one of our 4 GI / 4 reproductive sub-tissues without inflating apparent matches. Nerve - Tibial and the 2 non-anatomical "tissues" have no matching organ category.

In [10]:
panglao_raw <- read.delim(PANGLAODB_FILE, stringsAsFactors = FALSE, check.names = FALSE)
panglao_human <- panglao_raw %>% dplyr::filter(grepl('Hs', species))

panglao_t2g <- split(panglao_human$`official gene symbol`, panglao_human$`cell type`)
panglao_cell_to_organ <- panglao_human %>%
    dplyr::distinct(`cell type`, organ) %>%
    dplyr::group_by(`cell type`) %>%
    dplyr::summarise(organ = dplyr::first(organ), .groups = 'drop')

# 'GI tract' and 'Reproductive' are coarser than GTEx's split, but the user decided
# that's fine to use as shared ground truth for all of their constituent sub-tissues
# (same many-to-many principle already used for Heart's two sub-regions).
organ_to_tissue <- list(
    'Adrenal glands'   = 'Adrenal Gland',
    'Blood'            = 'Whole Blood',
    'Heart'            = c('Heart - Atrial Appendage', 'Heart - Left Ventricle'),
    'Kidney'           = 'Kidney - Cortex',
    'Liver'            = 'Liver',
    'Lungs'            = 'Lung',
    'Pancreas'         = 'Pancreas',
    'Skeletal muscle'  = 'Muscle - Skeletal',
    'Thyroid'          = 'Thyroid',
    'Vasculature'      = 'Artery - Tibial',
    'GI tract'         = c('Colon - Transverse', 'Esophagus - Mucosa',
                           'Small Intestine - Terminal Ileum', 'Stomach'),
    'Reproductive'     = c('Ovary', 'Prostate', 'Testis', 'Vagina')
)

all_gtex_tissues <- unique(shap_all$Tissue)
panglao_tissue_matched_names <- setNames(
    lapply(all_gtex_tissues, function(t) {
        organs_for_tissue <- names(organ_to_tissue)[vapply(organ_to_tissue, function(v) t %in% v, logical(1))]
        if (length(organs_for_tissue) == 0) return(character(0))
        panglao_cell_to_organ$`cell type`[panglao_cell_to_organ$organ %in% organs_for_tissue]
    }),
    all_gtex_tissues
)

# Three more tissues get a legitimate match by exact cell-type name rather than via the
# 'organ' column -- their organ labels are too broad (immune system holds every leukocyte
# type, not just B cells) or simply wrong for this purpose (Schwann cells are filed under
# 'Brain' in PanglaoDB, not under any peripheral-nerve category):
#  - Nerve - Tibial: 'Schwann cells' (peripheral nerve's defining resident glial cell).
#    'Peri-islet Schwann cells' stays excluded -- pancreas-context-specific, not generic.
#  - Cells - Cultured fibroblasts: 'Fibroblasts' (GTEx's sample is cultured skin fibroblasts).
#    'Myofibroblasts' stays excluded -- a distinct, contractile-activated subtype.
#  - Cells - EBV-transformed lymphocytes: EBV immortalizes mature B cells into
#    lymphoblastoid cell lines, so 'B cells'/'B cells memory'/'B cells naive'/'Plasma cells'.
EXTRA_PANGLAO_MATCHES <- list(
    'Nerve - Tibial' = 'Schwann cells',
    'Cells - Cultured fibroblasts' = 'Fibroblasts',
    'Cells - EBV-transformed lymphocytes' = c('B cells', 'B cells memory', 'B cells naive', 'Plasma cells')
)
for (t in names(EXTRA_PANGLAO_MATCHES)) {
    panglao_tissue_matched_names[[t]] <- union(panglao_tissue_matched_names[[t]], EXTRA_PANGLAO_MATCHES[[t]])
}

panglao_term2gene_df <- do.call(rbind, lapply(names(panglao_t2g), function(term) {
    data.frame(term = term, gene = panglao_t2g[[term]], stringsAsFactors = FALSE)
}))

panglao_assignment_df <- matched_names_to_df(panglao_tissue_matched_names)
write.csv(panglao_assignment_df, file.path(OUT_DIR, 'panglaodb_term_tissue_assignment.csv'), row.names = FALSE)

panglao_coverage <- data.frame(
    Tissue  = names(panglao_tissue_matched_names),
    N_terms = vapply(panglao_tissue_matched_names, length, integer(1))
) %>% dplyr::arrange(N_terms)
panglao_coverage

,Tissue,N_terms
,<chr>,<int>
Pituitary,Pituitary,0
Adrenal Gland,Adrenal Gland,1
Cells - Cultured fibroblasts,Cells - Cultured fibroblasts,1
Nerve - Tibial,Nerve - Tibial,1
Thyroid,Thyroid,1
Muscle - Skeletal,Muscle - Skeletal,3
Artery - Tibial,Artery - Tibial,4
Cells - EBV-transformed lymphocytes,Cells - EBV-transformed lymphocytes,4
Heart - Atrial Appendage,Heart - Atrial Appendage,4


In [11]:
panglao_ora_all <- run_resource_ora(panglao_term2gene_df, top_genes_per_lv, universe_genes)
write.csv(panglao_ora_all, file.path(OUT_DIR, 'panglaodb_ora_per_lv.csv'), row.names = FALSE)

results_panglaodb <- lapply(names(selections), function(nm) {
    run_alignment_summary(selections[[nm]], panglao_ora_all, panglao_tissue_matched_names,
                           file.path(OUT_DIR, 'panglaodb', nm))
})
names(results_panglaodb) <- names(selections)

dplyr::bind_rows(lapply(names(results_panglaodb), function(nm) {
    results_panglaodb[[nm]]$final_summary %>% dplyr::mutate(analysis = nm, .before = 1)
}))

analysis,n_tissues,n_tissues_with_coverage,n_tissues_correct,pct_tissue_correct,pct_tissue_correct_of_covered
<chr>,<int>,<int>,<int>,<dbl>,<dbl>
top1,23,22,14,60.86957,63.63636
cumulative25,23,22,18,78.26087,81.81818


## Combined comparison across all 3 resources

Per tissue, per analysis: how many of the 3 resources call it `tissue_correct` (only counting resources that actually have coverage for that tissue), plus a side-by-side final summary.

In [12]:
all_results <- list(CellMarker = results_cellmarker, C8 = results_c8, PanglaoDB = results_panglaodb)

combined_final <- dplyr::bind_rows(lapply(names(all_results), function(resource) {
    dplyr::bind_rows(lapply(names(selections), function(nm) {
        all_results[[resource]][[nm]]$final_summary %>%
            dplyr::mutate(resource = resource, analysis = nm, .before = 1)
    }))
}))
write.csv(combined_final, file.path(OUT_DIR, 'combined_final_summary.csv'), row.names = FALSE)
combined_final

resource,analysis,n_tissues,n_tissues_with_coverage,n_tissues_correct,pct_tissue_correct,pct_tissue_correct_of_covered
<chr>,<chr>,<int>,<int>,<int>,<dbl>,<dbl>
CellMarker,top1,23,21,12,52.17391,57.14286
CellMarker,cumulative25,23,21,15,65.21739,71.42857
C8,top1,23,17,14,60.86957,82.35294
C8,cumulative25,23,17,15,65.21739,88.23529
PanglaoDB,top1,23,22,14,60.86957,63.63636
PanglaoDB,cumulative25,23,22,18,78.26087,81.81818


In [13]:
build_combined_tissue_table <- function(analysis_name) {
    parts <- lapply(names(all_results), function(resource) {
        ts <- all_results[[resource]][[analysis_name]]$tissue_summary
        ts %>%
            dplyr::transmute(
                Tissue,
                !!resource := dplyr::case_when(
                    !has_coverage ~ NA,
                    tissue_correct ~ TRUE,
                    TRUE ~ FALSE
                )
            )
    })
    combined <- Reduce(function(a, b) dplyr::full_join(a, b, by = 'Tissue'), parts)
    combined$n_resources_correct <- rowSums(combined[, names(all_results)], na.rm = TRUE)
    combined$n_resources_with_coverage <- rowSums(!is.na(combined[, names(all_results)]))
    combined %>% dplyr::arrange(dplyr::desc(n_resources_correct))
}

combined_top1 <- build_combined_tissue_table('top1')
combined_cumulative25 <- build_combined_tissue_table('cumulative25')

write.csv(combined_top1, file.path(OUT_DIR, 'combined_tissue_table_top1.csv'), row.names = FALSE)
write.csv(combined_cumulative25, file.path(OUT_DIR, 'combined_tissue_table_cumulative25.csv'), row.names = FALSE)

cat('--- top1 ---\n')
print(combined_top1, n = Inf)
cat('\n--- cumulative25 ---\n')
print(combined_cumulative25, n = Inf)

--- top1 ---


# A tibble: 23 × 6
   Tissue  CellMarker C8    PanglaoDB n_resources_correct n_resources_with_cov…¹
   <chr>   <lgl>      <lgl> <lgl>                   <dbl>                  <dbl>
 1 Colon … TRUE       TRUE  TRUE                        3                      3
 2 Heart … TRUE       TRUE  TRUE                        3                      3
 3 Heart … TRUE       TRUE  TRUE                        3                      3
 4 Kidney… TRUE       TRUE  TRUE                        3                      3
 5 Liver   TRUE       TRUE  TRUE                        3                      3
 6 Lung    TRUE       TRUE  TRUE                        3                      3
 7 Pancre… TRUE       TRUE  TRUE                        3                      3
 8 Small … TRUE       TRUE  TRUE                        3                      3
 9 Stomach TRUE       TRUE  TRUE                        3                      3
10 Cells … FALSE      TRUE  TRUE                        2                      3
11 Esopha


--- cumulative25 ---


# A tibble: 23 × 6
   Tissue  CellMarker C8    PanglaoDB n_resources_correct n_resources_with_cov…¹
   <chr>   <lgl>      <lgl> <lgl>                   <dbl>                  <dbl>
 1 Cells … TRUE       TRUE  TRUE                        3                      3
 2 Colon … TRUE       TRUE  TRUE                        3                      3
 3 Heart … TRUE       TRUE  TRUE                        3                      3
 4 Heart … TRUE       TRUE  TRUE                        3                      3
 5 Kidney… TRUE       TRUE  TRUE                        3                      3
 6 Liver   TRUE       TRUE  TRUE                        3                      3
 7 Lung    TRUE       TRUE  TRUE                        3                      3
 8 Pancre… TRUE       TRUE  TRUE                        3                      3
 9 Small … TRUE       TRUE  TRUE                        3                      3
10 Stomach TRUE       TRUE  TRUE                        3                      3
11 Artery

## Liver sanity check across all 3 resources

In [14]:
lapply(names(all_results), function(resource) {
    cat('---', resource, '---\n')
    print(all_results[[resource]]$cumulative25$detail %>%
              dplyr::filter(Tissue == 'Liver') %>%
              dplyr::select(Tissue, LV, Rank, tissue_correct, n_true_terms, best_true_padj, matched_terms))
})
invisible(NULL)

--- CellMarker ---
# A tibble: 3 × 7
  Tissue LV     Rank tissue_correct n_true_terms best_true_padj matched_terms   
  <chr>  <chr> <int> <lgl>                 <int>          <dbl> <chr>           
1 Liver  LV21      1 TRUE                      2       7.59e- 9 Liver Bud Hepat…
2 Liver  LV455     2 TRUE                      2       5.15e-10 Liver Bud Hepat…
3 Liver  LV59      3 TRUE                      2       9.20e- 9 Hepatocyte Live…
--- C8 ---
# A tibble: 3 × 7
  Tissue LV     Rank tissue_correct n_true_terms best_true_padj matched_terms   
  <chr>  <chr> <int> <lgl>                 <int>          <dbl> <chr>           
1 Liver  LV21      1 TRUE                      7      1.52e- 96 AIZARANI_LIVER_…
2 Liver  LV455     2 TRUE                      6      1.46e- 66 AIZARANI_LIVER_…
3 Liver  LV59      3 TRUE                      5      1.06e-104 AIZARANI_LIVER_…
--- PanglaoDB ---
# A tibble: 3 × 7
  Tissue LV     Rank tissue_correct n_true_terms best_true_padj matched_terms
  <chr>  <

Tissue,LV,Rank,tissue_correct,n_true_terms,best_true_padj,matched_terms
<chr>,<chr>,<int>,<lgl>,<int>,<dbl>,<chr>
Liver,LV21,1,TRUE,2,7.594161e-09,Liver Bud Hepatic Cell Liver Human | Hepatocyte Liver Human
Liver,LV455,2,TRUE,2,5.145906e-10,Liver Bud Hepatic Cell Liver Human | Hepatocyte Liver Human
Liver,LV59,3,TRUE,2,9.198730e-09,Hepatocyte Liver Human | Liver Bud Hepatic Cell Liver Human
Tissue,LV,Rank,tissue_correct,n_true_terms,best_true_padj,matched_terms
<chr>,<chr>,<int>,<lgl>,<int>,<dbl>,<chr>
Liver,LV21,1,TRUE,7,1.515280e-96,AIZARANI_LIVER_C11_HEPATOCYTES_1 | DESCARTES_FETAL_LIVER_HEPATOBLASTS | AIZARANI_LIVER_C14_HEPATOCYTES_2 | AIZARANI_LIVER_C17_HEPATOCYTES_3 | AIZARANI_LIVER_C30_HEPATOCYTES_4 | AIZARANI_LIVER_C4_EPCAM_POS_BILE_DUCT_CELLS_1 | AIZARANI_LIVER_C31_KUPFFER_CELLS_5
Liver,LV455,2,TRUE,6,1.458683e-66,AIZARANI_LIVER_C14_HEPATOCYTES_2 | AIZARANI_LIVER_C11_HEPATOCYTES_1 | AIZARANI_LIVER_C17_HEPATOCYTES_3 | DESCARTES_FETAL_LIVER_HEPATOBLASTS | AIZARANI_LIVER_C30_HEPATOCYTES_4 | AIZARANI_LIVER_C31_KUPFFER_CELLS_5
Liver,LV59,3,TRUE,5,1.058871e-104,AIZARANI_LIVER_C14_HEPATOCYTES_2 | AIZARANI_LIVER_C11_HEPATOCYTES_1 | DESCARTES_FETAL_LIVER_HEPATOBLASTS | AIZARANI_LIVER_C17_HEPATOCYTES_3 | AIZARANI_LIVER_C30_HEPATOCYTES_4
Tissue,LV,Rank,tissue_correct,n_true_terms,best_true_padj,matched_terms


In [15]:
for (resource in names(all_results)) {
    for (nm in names(selections)) {
        out_dir <- file.path(OUT_DIR, tolower(resource), nm)
        stopifnot(file.exists(file.path(out_dir, 'alignment_detail.csv')))
        stopifnot(file.exists(file.path(out_dir, 'alignment_summary.csv')))
        stopifnot(file.exists(file.path(out_dir, 'alignment_final_pct.csv')))
    }
}
stopifnot(file.exists(file.path(OUT_DIR, 'combined_final_summary.csv')))
stopifnot(file.exists(file.path(OUT_DIR, 'combined_tissue_table_top1.csv')))
stopifnot(file.exists(file.path(OUT_DIR, 'combined_tissue_table_cumulative25.csv')))
cat('All checks passed.\n')

All checks passed.
